In [57]:
"""Content-based movie recommender: TF-IDF + cosine similarity."""
import json, re, pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel  # cosine sim for L2-normalised TF-IDF
 
PATH = "../../data/raw/movies_raw_enriched.csv"
KEEP = ["movie_id", "title", "overview", "genres", "keywords", "cast", "director"]
 
# ---------------- 1. EDA on the raw data ----------------
raw = pd.read_csv(PATH)
print("Raw shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes)
print("\nMissing values (raw):\n", raw.isna().sum())
print("\nDuplicate movie_id:", raw.movie_id.duplicated().sum(),
      "| duplicate titles:", raw.title.duplicated().sum())
 
# ---------------- 2. Keep 7 columns, THEN drop missing ----------------
df = raw[KEEP].copy()
print("\nAfter selecting 7 columns:", df.shape)
df = df.dropna().reset_index(drop=True)
print("After dropping missing values:", df.shape)   # expected 29820
print("Empty keyword lists:", (df.keywords == "[]").sum(),
      "| empty genre lists:", (df.genres == "[]").sum())
 
# ---------------- 3. Feature engineering ----------------
def parse_list(x):
    try:
        return json.loads(x)
    except Exception:
        return []
 
def squash(name):                      # "Mark Hamill" -> "markhamill" (keeps names as one token)
    return re.sub(r"[^a-z0-9]", "", name.lower())
 
def build_soup(r):
    genres = [squash(g) for g in parse_list(r.genres)]
    kws = [squash(k) for k in parse_list(r.keywords)]
    cast = [squash(c) for c in r.cast.split("|")[:5] if c.strip()]   # top 5 billed
    director = [squash(d) for d in r.director.split("|") if d.strip()]
    # overview kept as natural text; structured fields appended (director/genres repeated for weight)
    return " ".join([r.overview.lower()] + genres * 2 + kws + cast + director * 2)
 
df["soup"] = df.apply(build_soup, axis=1)
 
# ---------------- 4. TF-IDF ----------------
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                        min_df=2, max_features=100_000, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(df["soup"])
print("\nTF-IDF matrix:", tfidf_matrix.shape)
 
# NOTE: a full 29,820 x 29,820 cosine matrix is ~3.5 GB (float32), so similarity
# is computed per query against the sparse TF-IDF matrix instead. Results are identical.
title_to_idx = pd.Series(df.index, index=df.title.str.lower()).groupby(level=0).first()
 
def recommend(title, n=5):
    key = title.lower().strip()
    if key not in title_to_idx.index:
        close = [t for t in title_to_idx.index if key in t][:5]
        return f"'{title}' not found. Similar titles: {close}"
    i = title_to_idx[key]
    sims = linear_kernel(tfidf_matrix[i], tfidf_matrix).ravel()
    top = np.argsort(-sims)
    top = [j for j in top if j != i][:n]
    out = df.loc[top, ["movie_id", "title"]].copy()
    return out

Raw shape: (35001, 12)

Dtypes:
 movie_id          int64
title               str
overview            str
release_date        str
genres              str
keywords            str
runtime           int64
vote_average    float64
vote_count        int64
poster_url          str
cast                str
director            str
dtype: object

Missing values (raw):
 movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie_id: 0 | duplicate titles: 1665

After selecting 7 columns: (35001, 7)
After dropping missing values: (29820, 7)
Empty keyword lists: 16648 | empty genre lists: 2115

TF-IDF matrix: (29820, 100000)


In [59]:
recommend("The Godfather")

,movie_id,title
119,240,The Godfather Part II
121,242,The Godfather Part III
111,227,The Outsiders
114,232,Rumble Fish
17973,628339,Camera


In [26]:
"""Content-based movie recommender: TF-IDF + cosine similarity (v2)."""
import json, re, pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel  # cosine sim for L2-normalised TF-IDF
 
PATH = "../../data/raw/movies_raw_enriched.csv"
KEEP = ["movie_id", "title", "overview", "genres", "keywords", "cast", "director"]
 
# Feature weights: how many times each field is repeated in the text "soup"
W_GENRES, W_KEYWORDS, W_CAST, W_DIRECTOR = 1, 3, 3, 2
 
# ---------------- 1. EDA on the raw data ----------------
raw = pd.read_csv(PATH)
print("Raw shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes)
print("\nMissing values (raw):\n", raw.isna().sum())
print("\nDuplicate movie_id:", raw.movie_id.duplicated().sum(),
      "| duplicate titles:", raw.title.duplicated().sum())
 
# ---------------- 2. Keep 7 columns, THEN drop missing ----------------
df = raw[KEEP].copy()
print("\nAfter selecting 7 columns:", df.shape)
df = df.dropna().reset_index(drop=True)
print("After dropping missing values:", df.shape)   # expected 29820
 
# ---------------- 3. Feature engineering ----------------
def parse_list(x):
    try:
        return json.loads(x)
    except Exception:
        return []
 
def squash(name):                      # "Mark Hamill" -> "markhamill"
    return re.sub(r"[^a-z0-9]", "", name.lower())
 
def build_soup(r):
    genres = [squash(g) for g in parse_list(r.genres)]
    kws = [squash(k) for k in parse_list(r.keywords)]
    cast = [squash(c) for c in r.cast.split("|")[:5] if c.strip()]
    director = [squash(d) for d in r.director.split("|") if d.strip()]
    return " ".join([r.overview.lower()] + genres * W_GENRES + kws * W_KEYWORDS
                    + cast * W_CAST + director * W_DIRECTOR)
 
df["soup"] = df.apply(build_soup, axis=1)
 
# ---------------- 4. TF-IDF ----------------
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 1),   # unigrams only
                        min_df=2, max_features=100_000, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(df["soup"])
print("\nTF-IDF matrix:", tfidf_matrix.shape)
 
# Similarity is computed per query (full 29,820 x 29,820 matrix would be ~3.5 GB).
title_to_idx = pd.Series(df.index, index=df.title.str.lower()).groupby(level=0).first()
 
def recommend(title, n=5):
    key = title.lower().strip()
    if key not in title_to_idx.index:
        close = [t for t in title_to_idx.index if key in t][:5]
        return f"'{title}' not found. Similar titles: {close}"
    i = title_to_idx[key]
    sims = linear_kernel(tfidf_matrix[i], tfidf_matrix).ravel()
    top = [j for j in np.argsort(-sims) if j != i][:n]
    return df.loc[top, ["movie_id", "title"]].copy()
 

Raw shape: (35001, 12)

Dtypes:
 movie_id          int64
title               str
overview            str
release_date        str
genres              str
keywords            str
runtime           int64
vote_average    float64
vote_count        int64
poster_url          str
cast                str
director            str
dtype: object

Missing values (raw):
 movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie_id: 0 | duplicate titles: 1665

After selecting 7 columns: (35001, 7)
After dropping missing values: (29820, 7)

TF-IDF matrix: (29820, 56336)


In [37]:
recommend("The Notebook")

,movie_id,title
1920,12650,Il Mare
5312,61807,Ethan Frome
1933,12832,A Woman in Berlin
398,844,2046
14577,432836,Memoir of a Murderer


In [79]:
"""Content-based movie recommender: TF-IDF + cosine similarity (v2)."""
import json, re, pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel  # cosine sim for L2-normalised TF-IDF
 
PATH = "../../data/raw/movies_raw_enriched.csv"
KEEP = ["movie_id", "title", "overview", "genres", "keywords", "cast", "director"]
EXTRA = ["vote_count"]   # only used to re-rank results, NOT used in TF-IDF
 
# Feature weights: how many times each field is repeated in the text "soup"
W_GENRES, W_KEYWORDS, W_CAST, W_DIRECTOR = 1, 3, 3, 2
 
# ---------------- 1. EDA on the raw data ----------------
raw = pd.read_csv(PATH)
print("Raw shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes)
print("\nMissing values (raw):\n", raw.isna().sum())
print("\nDuplicate movie_id:", raw.movie_id.duplicated().sum(),
      "| duplicate titles:", raw.title.duplicated().sum())
 
# ---------------- 2. Keep 7 columns, THEN drop missing ----------------
df = raw[KEEP + EXTRA].copy()
print("\nAfter selecting 7 columns (+ vote_count):", df.shape)
df = df.dropna().reset_index(drop=True)
print("After dropping missing values:", df.shape)   # expected 29820
 
# ---------------- 3. Feature engineering ----------------
def parse_list(x):
    try:
        return json.loads(x)
    except Exception:
        return []
 
def squash(name):                      # "Mark Hamill" -> "markhamill"
    return re.sub(r"[^a-z0-9]", "", name.lower())
 
def build_soup(r):
    genres = [squash(g) for g in parse_list(r.genres)]
    kws = [squash(k) for k in parse_list(r.keywords)]
    cast = [squash(c) for c in r.cast.split("|")[:5] if c.strip()]
    director = [squash(d) for d in r.director.split("|") if d.strip()]
    return " ".join([r.overview.lower()] + genres * W_GENRES + kws * W_KEYWORDS
                    + cast * W_CAST + director * W_DIRECTOR)
 
df["soup"] = df.apply(build_soup, axis=1)
 
# ---------------- 4. TF-IDF ----------------
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 1),   # unigrams only
                        min_df=2, max_features=100_000, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(df["soup"])
print("\nTF-IDF matrix:", tfidf_matrix.shape)
 
# Similarity is computed per query (full 29,820 x 29,820 matrix would be ~3.5 GB).
title_to_idx = pd.Series(df.index, index=df.title.str.lower()).groupby(level=0).first()
 
def recommend(title, n=5, pool=50, min_votes=100):
    """Top `pool` by cosine similarity -> keep movies with >= min_votes -> sort by popularity-weighted score."""
    key = title.lower().strip()
    if key not in title_to_idx.index:
        close = [t for t in title_to_idx.index if key in t][:5]
        return f"'{title}' not found. Similar titles: {close}"
    i = title_to_idx[key]
    sims = linear_kernel(tfidf_matrix[i], tfidf_matrix).ravel()
    top = [j for j in np.argsort(-sims) if j != i][:pool]
    cand = df.loc[top, ["movie_id", "title", "vote_count"]].copy()
    cand["sim"] = sims[top]
    cand = cand[cand.vote_count >= min_votes]
    cand["score"] = cand.sim * np.log1p(cand.vote_count)
    return cand.sort_values("score", ascending=False).head(n)[["movie_id", "title"]]
 

Raw shape: (35001, 12)

Dtypes:
 movie_id          int64
title               str
overview            str
release_date        str
genres              str
keywords            str
runtime           int64
vote_average    float64
vote_count        int64
poster_url          str
cast                str
director            str
dtype: object

Missing values (raw):
 movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie_id: 0 | duplicate titles: 1665

After selecting 7 columns (+ vote_count): (35001, 8)
After dropping missing values: (29820, 8)

TF-IDF matrix: (29820, 56336)


In [82]:
"""Content-based movie recommender: TF-IDF + cosine similarity (v2)."""
import json, re, pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel  # cosine sim for L2-normalised TF-IDF
 
PATH = "../../data/raw/movies_raw_enriched.csv"
KEEP = ["movie_id", "title", "overview", "genres", "keywords", "cast", "director"]
EXTRA = ["vote_count"]   # only used to re-rank results, NOT used in TF-IDF
 
# Feature weights: how many times each field is repeated in the text "soup"
W_GENRES, W_KEYWORDS, W_CAST, W_DIRECTOR = 1, 3, 3, 2
 
# ---------------- 1. EDA on the raw data ----------------
raw = pd.read_csv(PATH)
print("Raw shape:", raw.shape)
print("\nDtypes:\n", raw.dtypes)
print("\nMissing values (raw):\n", raw.isna().sum())
print("\nDuplicate movie_id:", raw.movie_id.duplicated().sum(),
      "| duplicate titles:", raw.title.duplicated().sum())
 
# ---------------- 2. Keep 7 columns, THEN drop missing ----------------
df = raw[KEEP + EXTRA].copy()
print("\nAfter selecting 7 columns (+ vote_count):", df.shape)
df = df.dropna().reset_index(drop=True)
print("After dropping missing values:", df.shape)   # expected 29820
 
# ---------------- 3. Feature engineering ----------------
def parse_list(x):
    try:
        return json.loads(x)
    except Exception:
        return []
 
def squash(name):                      # "Mark Hamill" -> "markhamill"
    return re.sub(r"[^a-z0-9]", "", name.lower())
 
def build_soup(r):
    genres = [squash(g) for g in parse_list(r.genres)]
    kws = [squash(k) for k in parse_list(r.keywords)]
    cast = [squash(c) for c in r.cast.split("|")[:5] if c.strip()]
    director = [squash(d) for d in r.director.split("|") if d.strip()]
    return " ".join([r.overview.lower()] + genres * W_GENRES + kws * W_KEYWORDS
                    + cast * W_CAST + director * W_DIRECTOR)
 
df["soup"] = df.apply(build_soup, axis=1)
 
# ---------------- 4. TF-IDF ----------------
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 1),   # unigrams only
                        min_df=2, max_features=100_000, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(df["soup"])
print("\nTF-IDF matrix:", tfidf_matrix.shape)
 
# Similarity is computed per query (full 29,820 x 29,820 matrix would be ~3.5 GB).
title_to_idx = pd.Series(df.index, index=df.title.str.lower()).groupby(level=0).first()
 
def recommend(title, n=5, pool=50, min_votes=100):
    """Top `pool` by cosine similarity -> keep movies with >= min_votes -> sort by popularity-weighted score."""
    key = title.lower().strip()
    if key not in title_to_idx.index:
        close = [t for t in title_to_idx.index if key in t][:5]
        return f"'{title}' not found. Similar titles: {close}"
    i = title_to_idx[key]
    sims = linear_kernel(tfidf_matrix[i], tfidf_matrix).ravel()
    top = [j for j in np.argsort(-sims) if j != i][:pool]
    cand = df.loc[top, ["movie_id", "title", "vote_count"]].copy()
    cand["sim"] = sims[top]
    cand = cand[cand.vote_count >= min_votes]
    cand["score"] = cand.sim * np.log1p(cand.vote_count)
    return cand.sort_values("score", ascending=False).head(n)[["movie_id", "title"]]
 
# ---------------- 5. Save ----------------
with open("movie_recommender.pkl", "wb") as f:
    pickle.dump({"df": df.drop(columns=["soup"]), "tfidf": tfidf, "matrix": tfidf_matrix}, f)
 
if __name__ == "__main__":
    for t in ["The Notebook", "Star Wars", "The Dark Knight", "Toy Story"]:
        print(f"\n=== {t} ===")
        print(recommend(t).to_string(index=False))

Raw shape: (35001, 12)

Dtypes:
 movie_id          int64
title               str
overview            str
release_date        str
genres              str
keywords            str
runtime           int64
vote_average    float64
vote_count        int64
poster_url          str
cast                str
director            str
dtype: object

Missing values (raw):
 movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie_id: 0 | duplicate titles: 1665

After selecting 7 columns (+ vote_count): (35001, 8)
After dropping missing values: (29820, 8)

TF-IDF matrix: (29820, 56336)

=== The Notebook ===
 movie_id                title
    10229   A Walk to Remember
      844                 2046
    12650              Il Mare
   318846        The Big Short
      858 Sleepless in S